In [1]:
import pandas as pd

df = pd.read_csv("train.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [ ]:
df["Family_size"] = df["SibSp"] + df["Parch"] + 1
df["Family_size"].value_counts()
df.groupby("Family_size")["Survived"].mean()
# 四口之家存亡的平均年龄
print(df[df["Family_size"]== 4].groupby("Survived")["Age"].mean())


Survived
0    23.500000
1    16.972381
Name: Age, dtype: float64


In [83]:
print(df[df["Survived"] == 1].groupby("Sex")["Pclass"].value_counts())
print(df.groupby("Sex")["Survived"].mean())
df.groupby(["Sex", "Pclass"])["Survived"].mean()

Sex     Pclass
female  1         91
        3         72
        2         70
male    3         47
        1         45
        2         17
Name: count, dtype: int64
Sex
female    0.742038
male      0.188908
Name: Survived, dtype: float64


Sex     Pclass
female  1         0.968085
        2         0.921053
        3         0.500000
male    1         0.368852
        2         0.157407
        3         0.135447
Name: Survived, dtype: float64

In [ ]:
# df["Title"] = df["Name"].str.split(",").str[1].str.strip().str.split(".").str[0]
df["Title"] = df["Name"].str.extract(r",\s*([^\.]+)\.")
df["Title"].value_counts()

# title_mapping = {
#     "Mr": "Mr",
#     "Miss": "Miss",
#     "Mlle": "Miss",
#     "Mrs": "Mrs",
#     "Ms": "Mrs",
#     "Lady": "Mrs",
#     "the Countess": "Mrs",
#     "Mme": "Mrs",
#     "Master": "Master",
#     "Dr": "Rare",
#     "Rev": "Rare",
#     "Major": "Rare",
#     "Col": "Rare",
#     "Capt": "Rare",
#     "Sir": "Rare",
#     "Don": "Rare",
#     "Jonkheer": "Rare",
# }
# df["Title"] = df["Title"].replace(title_mapping)

df["Title"] = df["Title"].replace({"Mlle": "Miss"})
df.loc[df["Title"].isin(["Ms", "Lady", "the Countess", "Mme"]), "Title"] = "Mrs"
df.loc[df["Title"].isin(["Dr", "Rev", "Major", "Col", "Capt", "Sir", "Don", "Jonkheer"]), "Title"] = "Rare"
df["Title"].value_counts()


Title
Mr        517
Miss      184
Mrs       129
Master     40
Rare       21
Name: count, dtype: int64

In [52]:
df.groupby("Title")["Age"].median()

Title
Master     3.5
Miss      21.0
Mr        30.0
Mrs       35.0
Rare      49.0
Name: Age, dtype: float64

In [ ]:
df["Age"] = df["Age"].fillna(df.groupby("Title")["Age"].transform("median"))
df["Age"].isna().sum()
df["Age"].describe()

count    891.000000
mean      29.393008
std       13.269209
min        0.420000
25%       21.000000
50%       30.000000
75%       35.000000
max       80.000000
Name: Age, dtype: float64

In [ ]:
df["Has_cabin"] = df["Cabin"].notna()
print(df["Has_cabin"].head(10))

0    False
1     True
2    False
3     True
4    False
5    False
6     True
7    False
8    False
9    False
Name: has_cabin, dtype: bool


In [3]:
print(df["Ticket"].head(20))

0            A/5 21171
1             PC 17599
2     STON/O2. 3101282
3               113803
4               373450
5               330877
6                17463
7               349909
8               347742
9               237736
10             PP 9549
11              113783
12           A/5. 2151
13              347082
14              350406
15              248706
16              382652
17              244373
18              345763
19                2649
Name: Ticket, dtype: str


In [4]:
print(df["Ticket"].nunique())

681


In [28]:
cabin_notna_by_ticket = df.groupby("Ticket")["Cabin"].apply(lambda x: x.notna().sum())

all_missing = (cabin_notna_by_ticket == 0).sum()
total_groups = len(cabin_notna_by_ticket)

print(f"全缺分组数: {all_missing}")
print(f"总分组数: {total_groups}")
print(f"全缺占比: {all_missing / total_groups:.1%}")

全缺分组数: 539
总分组数: 681
全缺占比: 79.1%


Summary of Findings:

 Survival strongly correlates with gender and class. Females altogether have better survival rate than male. Out of which, female from the first class has the highest survival rate, and even female from the lowest third class (50.0%) still have more survival rate than male from the first class (36.9%), with male from the third class having the least chance of survival, suggesting gender is stronger predictor than wealth/class in this dataframe.
 
 Within the newly created column "Family", and found that the family of four has the highest chance of survival out of all family groups. Family sizes of 8 and 11 showed 0% survival, but these groups only had 6-7 passengers each, so this extreme result is likely due to small sample size rather than a strong pattern. 
 
 The number of missing cabin information falls under the “Tickets" group is  539, ie. 539 out of 681 groups of tickets have zero cabin information. Attempting to fillin cabin values using shared Tickets numbers proved largely ineffective. A new column "has_cabin" was created instead, which could carry predictive signal tied to socioeconomic status.

Methodology:
1. Examine the columns info of the .csv file, and check the percentage of missing data for the columns, ie. "Age" and "Cabin" columns has significant number of data missing.

2. Using Tickets information to group and check if the same tickets group could apply auto fill of the cabin information. 

3. Categorize passengers by title extracted from Name. Title could tell a lot of information, usually the ones with the special titles enjoys a higher social rank status, and Master refers to little boys, and separate Miss to Mrs, and after all the grouping is done, use the group median age to backfill in the missing info.

4. Created a new column called "Family" with adding sibling, parents and children number together, to see the percentage of the survival for different kinds of family. 

5. Adopted .nunique() to check the tickets grouping and have the result of 681 which showcase that out of 891, there were multiple people sharing the same tickects information. Cross check "cabin" under "tickets" category, and there are 0,1,2,3 or more possible entries per group. Test exactly how many groups of tickets that have zero cabin information, and the number is too big to worth diving deeper, need to seek alternative ways.